# GEPA — Genetic-Pareto Prompt Evolution

**Week 6 | Notebook 5 of 6**

**What you'll learn:**
- GEPA vs MIPROv2 vs RL — conceptual comparison
- Setting up GEPA on a structured extraction task
- Visualizing the Pareto frontier (quality vs. cost)
- Running GEPA on AIME-style math problems
- Population evolution across generations
- Comparing GEPA-optimized vs MIPROv2-optimized programs

**Runtime:** ~90 minutes (computationally intensive)

**Cost-saving:** Reduced population size and generations for demo.

In [ ]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/05_gepa_optimizer.ipynb")

## 1. Setup

In [ ]:
import dspy
import matplotlib.pyplot as plt
from dspy.teleprompt import GEPA, MIPROv2

from src.config import get_dspy_lm
from src.datasets import generate_math_problems

lm = get_dspy_lm()
dspy.configure(lm=lm)

print("✅ DSPy configured for GEPA optimization")

## 2. GEPA vs MIPROv2 vs RL — Conceptual Comparison

In [ ]:
comparison = """
| Dimension | BootstrapFewShot | MIPROv2 | GEPA | RL |
|-----------|------------------|---------|------|-----|
| Strategy | Few-shot demos | Bayesian opt | Genetic algorithm | Gradient ascent |
| Search space | Demos only | Instructions + demos | Prompt population | Policy weights |
| Requires differentiable reward | No | No | No | Yes |
| Handles multi-objective | No | Partial | Yes (Pareto) | No |
| Best for | Quick baseline | General tasks | Complex, diverse tasks | Simple tasks |
| Cost | Low | Medium | High | Very high |
"""

print(comparison)

print("\n🧬 GEPA Key Features:")
print("  • Maintains population of prompt candidates")
print("  • Mutates and recombines using LLM reflections")
print("  • Pareto frontier balances quality vs cost/length")
print("  • No differentiable reward needed")

## 3. Task Setup — Math Problem Solving

In [ ]:
class MathProblem(dspy.Signature):
    """Solve math word problems step by step."""

    problem: str = dspy.InputField()
    solution: str = dspy.OutputField()
    answer: str = dspy.OutputField(desc="Final numeric answer only")


class MathSolver(dspy.Module):
    def __init__(self):
        super().__init__()
        self.solve = dspy.ChainOfThought(MathProblem)

    def forward(self, problem):
        return self.solve(problem=problem)


# Prepare data
math_data = generate_math_problems(30)
examples = [
    dspy.Example(problem=d["question"], solution=d["answer"], answer=d["answer"]).with_inputs(
        "problem"
    )
    for d in math_data
]

trainset = examples[:20]
devset = examples[20:]


def math_metric(example, prediction, trace=None):
    """Check if answer matches."""
    return 1.0 if example.answer.strip() in prediction.answer.strip() else 0.0


print(f"Train: {len(trainset)}, Dev: {len(devset)}")

## 4. Running GEPA

In [ ]:
# GEPA with reduced parameters for demo
gepa = GEPA(
    metric=math_metric,
    population_size=5,  # Reduced from 20
    generations=3,  # Reduced from 10
)

print("Running GEPA optimization...")
print("This may take 10-20 minutes with reduced settings.")
print("For full power: population_size=20, generations=10")

gepa_optimized = gepa.compile(
    MathSolver(),
    trainset=trainset,
)

print("\n✅ GEPA optimization complete!")

## 5. Evaluating GEPA Results

In [ ]:
from dspy.evaluate import Evaluate

evaluator = Evaluate(devset=devset, metric=math_metric, num_threads=2, display_progress=True)

# Baseline
baseline = MathSolver()
baseline_score = evaluator(baseline)

# GEPA optimized
gepa_score = evaluator(gepa_optimized)

print(f"\nBaseline score: {baseline_score:.2f}")
print(f"GEPA score: {gepa_score:.2f}")
print(f"Improvement: {gepa_score - baseline_score:+.2f}")

## 6. Comparing with MIPROv2

In [ ]:
# Run MIPROv2 on same task for comparison
mipro = MIPROv2(metric=math_metric, num_candidates=3)

mipro_optimized = mipro.compile(MathSolver(), trainset=trainset, num_trials=5, valset=devset)

mipro_score = evaluator(mipro_optimized)

print("\nComparison:")
print(f"  Baseline:     {baseline_score:.2f}")
print(f"  MIPROv2:      {mipro_score:.2f}")
print(f"  GEPA:         {gepa_score:.2f}")

if gepa_score > mipro_score:
    print(f"\n🏆 GEPA wins by {gepa_score - mipro_score:+.2f}!")
else:
    print(f"\n📊 MIPROv2 wins by {mipro_score - gepa_score:+.2f}")
    print("   Try increasing GEPA population/generations.")

## 7. Visualizing Pareto Frontier

In [ ]:
# Simulated Pareto frontier data
# In real GEPA, you would extract this from the optimization history
import numpy as np

# Mock data: (cost, quality) pairs
pareto_points = np.array(
    [
        [0.5, 0.60],
        [0.7, 0.72],
        [1.0, 0.80],
        [1.3, 0.85],
        [1.8, 0.88],
        [2.5, 0.90],
    ]
)

plt.figure(figsize=(8, 6))
plt.plot(pareto_points[:, 0], pareto_points[:, 1], "bo-", linewidth=2, markersize=8)
plt.fill_between(pareto_points[:, 0], pareto_points[:, 1], alpha=0.3)
plt.xlabel("Cost (API calls / tokens)", fontsize=12)
plt.ylabel("Quality (task score)", fontsize=12)
plt.title("GEPA Pareto Frontier: Quality vs Cost", fontsize=14)
plt.grid(True, alpha=0.3)
plt.annotate(
    "Selected",
    xy=pareto_points[3],
    xytext=(2.0, 0.75),
    arrowprops={"arrowstyle": "->", "color": "red"},
    fontsize=11,
    color="red",
)
plt.tight_layout()
plt.show()

print("\n📊 Pareto frontier shows tradeoff between quality and cost.")
print("   GEPA automatically selects optimal points.")

## 8. Exercise: Apply GEPA to Your Enterprise Task

Run GEPA on your own structured extraction or reasoning task:
1. Define the task and metric
2. Collect 50+ training examples
3. Run GEPA with population_size=20, generations=10
4. Compare against MIPROv2 baseline
5. Inspect the evolved prompts

In [ ]:
# YOUR TURN: GEPA on your domain

# class MyTask(dspy.Signature):
#     ...

# gepa = GEPA(
#     metric=my_metric,
#     population_size=20,
#     generations=10
# )

# optimized = gepa.compile(MyModule(), trainset=trainset)
# optimized.save("gepa_optimized.json")

---

**Next:** [06_finetuning.ipynb](06_finetuning.ipynb) — BetterTogether finetuning pipeline